# Langkah 4 - Fine-tuning IndoBERT (Google Colab, GPU)

Jalankan di Colab: **Runtime > Change runtime type > T4 GPU**, lalu **Runtime > Run all**.

Saat diminta, unggah tiga berkas dari folder `kode/data/processed/`:
`train.csv`, `val.csv`, dan `test_kondisi.csv`.
Setelah selesai, berkas hasil akan terunduh otomatis. Simpan ke folder `kode/hasil/`.

In [2]:
import os, sys, platform, random, time, json
import numpy as np, pandas as pd, torch, transformers
print('Python', platform.python_version(), '| torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'TIDAK ADA - aktifkan GPU dulu')

Python 3.13.16 | torch 2.14.1+cu130 | transformers 5.22.1
GPU: Tesla T4


In [3]:
# Unggah data (dilewati jika berkas sudah ada di /content)
BERKAS = ['train.csv', 'val.csv', 'test_kondisi.csv']
if not all(os.path.exists(b) for b in BERKAS):
    from google.colab import files
    files.upload()
assert all(os.path.exists(b) for b in BERKAS), 'Berkas belum lengkap'

Saving test_kondisi.csv to test_kondisi.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


In [4]:
# Konfigurasi fine-tuning
CHECKPOINT   = 'indobenchmark/indobert-base-p1'
SEED         = 42
LR           = 1e-5
BATCH        = 16
MAX_EPOCH    = 4
MAX_LEN      = 128
WEIGHT_DECAY = 0.01
WARMUP_RASIO = 0.1
CLIP_NORM    = 1.0
PATIENCE     = 2      # early stopping berdasarkan validation loss
FP16         = torch.cuda.is_available()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
# Prapemrosesan minimal (sama dengan praproses.py)
import re
_KECIL = str.maketrans('ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz')
_SPASI = re.compile(r'\s+')
def praproses(t): return _SPASI.sub(' ', t.translate(_KECIL)).strip()

baca = lambda f: pd.read_csv(f, dtype={'komentar': str}, keep_default_na=False)
train, val = baca('train.csv'), baca('val.csv')
uji = pd.read_csv('test_kondisi.csv', keep_default_na=False)
train['teks'] = train.komentar.map(praproses); val['teks'] = val.komentar.map(praproses)
print(len(train), len(val), len(uji))

5154 1105 1105


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_fscore_support

tok = AutoTokenizer.from_pretrained(CHECKPOINT)

def encode(teks, label=None):
    e = tok(list(teks), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
    t = [e['input_ids'], e['attention_mask']]
    if label is not None: t.append(torch.tensor(list(label)))
    return TensorDataset(*t)

g = torch.Generator().manual_seed(SEED)
dl_train = DataLoader(encode(train.teks, train.label), batch_size=BATCH, shuffle=True, generator=g)
dl_val   = DataLoader(encode(val.teks, val.label), batch_size=64)
panjang = [len(tok.tokenize(t)) for t in train.teks]
print('Komentar latih yang terpotong (>%d token):' % (MAX_LEN - 2), sum(p > MAX_LEN - 2 for p in panjang))

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Komentar latih yang terpotong (>126 token): 0


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2).to(device)
no_decay = ['bias', 'LayerNorm.weight']
grup = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
opt = torch.optim.AdamW(grup, lr=LR)
total_step = len(dl_train) * MAX_EPOCH
sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RASIO * total_step), total_step)
scaler = torch.amp.GradScaler('cuda', enabled=FP16)

@torch.no_grad()
def prediksi(dl):
    model.eval(); semua = []
    for b in dl:
        with torch.autocast('cuda', dtype=torch.float16, enabled=FP16):
            logit = model(input_ids=b[0].to(device), attention_mask=b[1].to(device)).logits
        semua.append(logit.float().cpu())
    return torch.cat(semua)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
log, terbaik, loss_terbaik, tanpa_perbaikan, state_terbaik = [], -1.0, float('inf'), 0, None
t_mulai = time.time()
for epoch in range(1, MAX_EPOCH + 1):
    model.train(); total_loss = 0.0
    for ids, mask, y in dl_train:
        opt.zero_grad()
        with torch.autocast('cuda', dtype=torch.float16, enabled=FP16):
            out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device))
        scaler.scale(out.loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(opt); scaler.update(); sched.step()
        total_loss += out.loss.item()
    logit = prediksi(dl_val)
    val_loss = torch.nn.functional.cross_entropy(logit, torch.tensor(val.label.values)).item()
    pv = logit.argmax(1).numpy()
    p, r, f, _ = precision_recall_fscore_support(val.label, pv, pos_label=1, average='binary')
    log.append(dict(epoch=epoch, train_loss=total_loss / len(dl_train), val_loss=val_loss,
                    val_precision=p, val_recall=r, val_f1=f, detik=time.time() - t_mulai))
    print(log[-1])
    # checkpoint: simpan bobot pada epoch dengan F1 validasi tertinggi
    if f > terbaik:
        terbaik = f
        state_terbaik = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        log[-1]['dipilih'] = True
    # early stopping: berhenti jika validation loss tidak turun selama PATIENCE epoch
    if val_loss < loss_terbaik:
        loss_terbaik, tanpa_perbaikan = val_loss, 0
    else:
        tanpa_perbaikan += 1
        if tanpa_perbaikan >= PATIENCE:
            print('Early stopping'); break
model.load_state_dict(state_terbaik)
pd.DataFrame(log).to_csv('indobert_log.csv', index=False)
print('F1 validasi terbaik:', terbaik)

{'epoch': 1, 'train_loss': 0.20296913703713254, 'val_loss': 0.08875644207000732, 'val_precision': 0.9595959595959596, 'val_recall': 0.9359605911330049, 'val_f1': 0.9476309226932669, 'detik': 45.178794384002686}
{'epoch': 2, 'train_loss': 0.036617190130170285, 'val_loss': 0.07362192869186401, 'val_precision': 0.9842105263157894, 'val_recall': 0.9211822660098522, 'val_f1': 0.9516539440203562, 'detik': 89.6058702468872}
{'epoch': 3, 'train_loss': 0.020826955144250356, 'val_loss': 0.09911301732063293, 'val_precision': 0.9455445544554455, 'val_recall': 0.9408866995073891, 'val_f1': 0.9432098765432099, 'detik': 134.75387001037598}
{'epoch': 4, 'train_loss': 0.0119466691194304, 'val_loss': 0.09354271739721298, 'val_precision': 0.955, 'val_recall': 0.9408866995073891, 'val_f1': 0.9478908188585607, 'detik': 179.92068457603455}
Early stopping
F1 validasi terbaik: 0.9516539440203562


In [9]:
# Prediksi tiga kondisi pada data uji
out = uji[['id_asli', 'label', 'masuk_evaluasi']].copy()
for k in ['normal', 'unicode', 'angka']:
    logit = prediksi(DataLoader(encode(uji[f'teks_{k}']), batch_size=64))
    prob = torch.softmax(logit, 1)[:, 1].numpy()
    out[f'pred_{k}'] = (prob >= 0.5).astype(int)
    out[f'prob_{k}'] = prob
out.to_csv('pred_indobert.csv', index=False)

from sklearn.metrics import confusion_matrix
ev = out[out.masuk_evaluasi.astype(str) == 'True']
for k in ['normal', 'unicode', 'angka']:
    print(k, confusion_matrix(ev.label, ev[f'pred_{k}']).ravel(), '(tn fp fn tp)')

normal [898   4   3 167] (tn fp fn tp)
unicode [898   4  34 136] (tn fp fn tp)
angka [898   4  28 142] (tn fp fn tp)


In [10]:
# Catat lingkungan eksekusi, lalu unduh hasil
info = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
            fp16=FP16, seed=SEED, lr=LR, batch=BATCH, max_epoch=MAX_EPOCH, max_len=MAX_LEN,
            weight_decay=WEIGHT_DECAY, warmup=WARMUP_RASIO, clip=CLIP_NORM, patience=PATIENCE,
            epoch_dijalankan=len(log), f1_val_terbaik=terbaik, total_detik=time.time() - t_mulai)
json.dump(info, open('indobert_info.json', 'w'), indent=2)
print(info)
try:
    from google.colab import files
    for f in ['pred_indobert.csv', 'indobert_log.csv', 'indobert_info.json']:
        files.download(f)
except ImportError:
    pass

{'python': '3.13.16', 'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'gpu': 'Tesla T4', 'fp16': True, 'seed': 42, 'lr': 1e-05, 'batch': 16, 'max_epoch': 4, 'max_len': 128, 'weight_decay': 0.01, 'warmup': 0.1, 'clip': 1.0, 'patience': 2, 'epoch_dijalankan': 4, 'f1_val_terbaik': 0.9516539440203562, 'total_detik': 186.58903908729553}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>